In [4]:
import kagglehub

kagglehub.login()

In [5]:
path = kagglehub.competition_download("cifar-10")

path

'/root/.cache/kagglehub/competitions/cifar-10'

In [6]:
%pip install py7zr
import os
import py7zr

# 定義要解壓的 7z 檔案路徑
train_7z = os.path.join(path, "train.7z")
test_7z = os.path.join(path, "test.7z")

# 解壓 train.7z
if os.path.exists(train_7z):
    print("正在解壓 train.7z...")
    with py7zr.SevenZipFile(train_7z, mode='r') as archive:
        archive.extractall(path=path)
    print("train.7z 解壓完成！")

# 解壓 test.7z
if os.path.exists(test_7z):
    print("正在解壓 test.7z...")
    with py7zr.SevenZipFile(test_7z, mode='r') as archive:
        archive.extractall(path=path)
    print("test.7z 解壓完成！")

正在解壓 train.7z...
train.7z 解壓完成！
正在解壓 test.7z...
test.7z 解壓完成！


In [7]:
import pandas as pd

train_labels = path+'/trainLabels.csv'

raw_csv = pd.read_csv(train_labels)
labels = dict(zip(raw_csv.iloc[:, 0], raw_csv.iloc[:, 1]))

tags = list(set(labels.values()))
tags

['deer',
 'truck',
 'cat',
 'bird',
 'ship',
 'dog',
 'horse',
 'airplane',
 'automobile',
 'frog']

In [8]:
import shutil

for tag in tags:
    os.makedirs(path+'/train/'+tag)
for img in os.listdir(path+'/train'):
    if img.endswith('.png'):
        shutil.move(
            path+'/train/'+img, 
            path+'/train/'+labels[int(img[:-4])]+'/'+img
        )

os.makedirs(path+'/test/unknown/')
for img in os.listdir(path+'/test'):
    if img.endswith('.png'):
        shutil.move(
            path+'/test/'+img, 
            path+'/test/unknown/'+img
        )

FileExistsError: [Errno 17] File exists: '/root/.cache/kagglehub/competitions/cifar-10/train/deer'

In [ ]:
import torch
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split


train_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomHorizontalFlip(),  
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

valid_transform = transforms.Compose([
    transforms.Resize((32, 32)),      
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset_full = datasets.ImageFolder(root=path+'/train', transform=train_transform)
valid_dataset_full = datasets.ImageFolder(root=path+'/train', transform=valid_transform)

indices = list(range(len(train_dataset_full)))
labels = train_dataset_full.targets

train_idx, valid_idx = train_test_split(
    indices, 
    test_size=0.2, 
    stratify=labels, 
    random_state=42
)

train_dataset = Subset(train_dataset_full, train_idx)
valid_dataset = Subset(valid_dataset_full, valid_idx)
test_dataset = datasets.ImageFolder(root=path+'/test', transform=train_transform)


In [ ]:
import torch
import torchvision.models as models
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

weights = models.ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)

for param in model.parameters():
    param.requires_grad = False
num_classes = len(tag)
model.fc = nn.Linear(model.fc.in_features, num_classes)

# * 一定要在fc層裝好之後再送 這樣才會
model = model.to(device)

Using device: cuda


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
import torch.optim as optim

lr, batch_size, num_epochs = 0.001, 128, 10


test_iter = DataLoader(test_dataset, batch_size)
train_iter = DataLoader(train_dataset, batch_size)
valid_iter = DataLoader(valid_dataset, batch_size)

optimizer = optim.Adam(model.fc.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

total_loss, total_acc = {"train":[], "valid": []}, {"train":[], "valid": []}

for epoch in range(num_epochs):
    model.train()
    train_loss, train_acc = 0.0, 0

    for input, label in train_iter:
        input, label = input.to(device), label.to(device)
        optimizer.zero_grad()
        hat_y = model(input)

        loss = criterion(hat_y, label)
        loss.backward()

        optimizer.step()
    
        _, preds = torch.max(hat_y, 1)
        train_loss += loss.item() * input.size(0)
        train_acc  += int(torch.sum(preds == label.data))

    total_loss["train"].append(train_loss / len(train_dataset))
    total_acc["train"].append(train_acc / len(train_dataset))

    model.eval()
    valid_loss, valid_acc = 0.0, 0

    with torch.no_grad():
        for inputs, label in valid_iter:
            inputs, label = inputs.to(device), label.to(device)

            hat_y = model(inputs)
            loss = criterion(hat_y, label)

            _, preds = torch.max(hat_y, 1)
            valid_loss += loss.item() * inputs.size(0)
            valid_acc  += int(torch.sum(preds == label.data))
    
    total_loss["valid"].append(valid_loss / len(valid_dataset))
    total_acc["valid"].append(valid_acc / len(valid_dataset))
    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {total_loss['train'][-1]:.4f} "
        f"Acc: {total_acc['train'][-1]:.4f} | "
        f"Valid Loss: {total_loss['valid'][-1]:.4f} "
        f"Acc: {total_acc['valid'][-1]:.4f}"
    )


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
import matplotlib.pyplot as plt

plt.plot(total_loss["valid"], range(1, len(total_loss["valid"])+1), label='valid')
plt.plot(total_loss["train"], range(1, len(total_loss["train"])+1), label='train')

In [ ]:
from google.colab import drive
import shutil
import os

# 1. 掛載 Google Drive 到 Colab 虛擬機
# 執行後會跳出視窗請求權限，請點選「連線到 Google 雲端硬碟」並允許
drive.mount('/content/drive')

# 2. 定義來源檔案路徑與 Google Drive 的目標儲存路徑
source_path = 'vgg_kfold_submission.csv'
# 預設儲存於雲端硬碟的根目錄，您也可以自訂資料夾路徑如 '/content/drive/MyDrive/Colab Notebooks/'
target_path = '/content/drive/MyDrive/'

# 3. 檢查檔案是否存在並執行複製上傳
if os.path.exists(source_path):
    shutil.copy(source_path, target_path)
    print(f"🎉 上傳成功！檔案已儲存至 Google Drive 根目錄，檔名為: vgg_kfold_submission.csv")
else:
    print(f"❌ 找不到來源檔案 {source_path}，請確認先前的預測程式碼是否有成功產生 CSV 檔。")